Practical HW 1
Machine Learning 2
April 13 2026
Amanda Boschman

#Practical Homework 1: Decision Trees

Purpose of this notebook/assignment/goal/some description.

Research Question: How well can youth demographics, family life, and experiences predict alcohol usage in a month period, whether the youth has ever used marijuana, and cigarette usage frequency?

Data description here

Target Outcomes:
1. Binary Classification: MRJFLAG (ever used marijuana)
2. Multi-Class Classification: ALCMDAYS (# of days used alcohol in past month)
3. Regression Classification: IRCIGFM (cigarette frequency in past month)


Predictors:
- IFATHER (categorical)
- POVERTY3 (categorical)
- IRSEX (binary)
- NEWRACE2 (categorical)
- PRCHORE2 (binary)
- PRLMTTV2 (binary)
- PARCHKHW (binary)
- YFFLTMRJ2 (binary)
- RLGATTD (binary)
- AVGGRADE (binary)
- IRALCAGE (numeric? categorical?)

Describe each variable and what their possible values mean like IRALCAGE value 991 meaning.


Imports:

In [24]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree, DecisionTreeRegressor
from sklearn.preprocessing import OneHotEncoder, LabelBinarizer
from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.inspection import PartialDependenceDisplay

Load Data

In [4]:
youth_data = pd.read_csv('/Users/amandaboschman/Desktop/ML2 Spring 2026/youth_data.csv')

Explore Data

In [5]:
youth_data.head()

,IRALCFY,IRMJFY,IRCIGFM,IRSMKLSS30N,IRALCFM,IRMJFM,IRCIGAGE,IRSMKLSSTRY,IRALCAGE,IRMJAGE,...,EDUSCHLGO,EDUSCHGRD2,EDUSKPCOM,IMOTHER,IFATHER,INCOME,GOVTPROG,POVERTY3,PDEN10,COUTYP4
0,991,991,91.0,91,91.0,91.0,991,991,991,991,...,1,3,0,1,1,2,2,1,2,2
1,991,60,91.0,91,91.0,2.0,991,991,991,14,...,1,6,0,1,1,2,2,1,2,2
2,1,991,91.0,91,93.0,91.0,991,991,11,991,...,1,2,1,1,1,4,2,3,1,1
3,991,991,91.0,91,91.0,91.0,991,991,991,991,...,1,2,0,1,1,3,2,2,1,1
4,991,991,91.0,91,91.0,91.0,991,991,991,991,...,1,5,0,1,1,4,2,3,2,2


In [6]:
youth_data.columns

Index(['IRALCFY', 'IRMJFY', 'IRCIGFM', 'IRSMKLSS30N', 'IRALCFM', 'IRMJFM',
       'IRCIGAGE', 'IRSMKLSSTRY', 'IRALCAGE', 'IRMJAGE', 'MRJFLAG', 'ALCFLAG',
       'TOBFLAG', 'ALCYDAYS', 'MRJYDAYS', 'ALCMDAYS', 'MRJMDAYS', 'CIGMDAYS',
       'SMKLSMDAYS', 'SCHFELT', 'TCHGJOB', 'AVGGRADE', 'STNDSCIG', 'STNDSMJ',
       'STNDALC', 'STNDDNK', 'PARCHKHW', 'PARHLPHW', 'PRCHORE2', 'PRLMTTV2',
       'PARLMTSN', 'PRGDJOB2', 'PRPROUD2', 'ARGUPAR', 'YOFIGHT2', 'YOGRPFT2',
       'YOHGUN2', 'YOSELL2', 'YOSTOLE2', 'YOATTAK2', 'PRPKCIG2', 'PRMJEVR2',
       'PRMJMO', 'PRALDLY2', 'YFLPKCG2', 'YFLTMRJ2', 'YFLMJMO', 'YFLADLY2',
       'FRDPCIG2', 'FRDMEVR2', 'FRDMJMON', 'FRDADLY2', 'TALKPROB', 'PRTALK3',
       'PRBSOLV2', 'PREVIOL2', 'PRVDRGO2', 'GRPCNSL2', 'PREGPGM2', 'YTHACT2',
       'DRPRVME3', 'ANYEDUC3', 'RLGATTD', 'RLGIMPT', 'RLGDCSN', 'RLGFRND',
       'IRSEX', 'NEWRACE2', 'HEALTH2', 'EDUSCHLGO', 'EDUSCHGRD2', 'EDUSKPCOM',
       'IMOTHER', 'IFATHER', 'INCOME', 'GOVTPROG', 'POVERTY3', 'PDEN10',

In [25]:
youth_data.shape

(10561, 79)

Selected Target Variables

In [23]:
bin_target = 'MRJFLAG'

multi_target = 'ALCMDAYS'

reg_target = 'IRCIGFM'

Selected Predictor Variables

In [18]:
predictors = ['IFATHER', 'POVERTY3', 'IRSEX', 'NEWRACE2', 'PRCHORE2', 'PRLMTTV2', 'PARCHKHW', 'YFLTMRJ2', 'RLGATTD', 'AVGGRADE', 'IRALCAGE']

In [19]:
youth_data.nunique()[predictors]

IFATHER      3
POVERTY3     3
IRSEX        2
NEWRACE2     7
PRCHORE2     2
PRLMTTV2     2
PARCHKHW     2
YFLTMRJ2     2
RLGATTD      2
AVGGRADE     2
IRALCAGE    18
dtype: int64

In [20]:
for col in predictors:
    print(f'{col}')
    print(youth_data[col].unique())

IFATHER
[1 2 3]
POVERTY3
[1 3 2]
IRSEX
[1 2]
NEWRACE2
[7 1 6 4 2 5 3]
PRCHORE2
[ 1.  2. nan]
PRLMTTV2
[ 2.  1. nan]
PARCHKHW
[ 1.  2. nan]
YFLTMRJ2
[ 1.  2. nan]
RLGATTD
[ 2.  1. nan]
AVGGRADE
[ 2. nan  1.]
IRALCAGE
[991  11  12  14  15  16  13  17   6  10   8   7   9   5   4   3   2   1]


Data Cleaning & Preparation

Create DataFrame of Selected Predictors & Target Outcomes

In [30]:
selected_cols = predictors + [bin_target] + [multi_target] + [reg_target]

df = youth_data[selected_cols]

df.head()

,IFATHER,POVERTY3,IRSEX,NEWRACE2,PRCHORE2,PRLMTTV2,PARCHKHW,YFLTMRJ2,RLGATTD,AVGGRADE,IRALCAGE,MRJFLAG,ALCMDAYS,IRCIGFM
0,1,1,1,7,1.0,2.0,1.0,1.0,2.0,2.0,991,0,5,91.0
1,1,1,2,1,2.0,2.0,1.0,1.0,2.0,2.0,991,1,5,91.0
2,1,3,1,6,1.0,1.0,1.0,1.0,2.0,2.0,11,0,5,91.0
3,1,2,2,7,2.0,2.0,1.0,1.0,2.0,NaN,991,0,5,91.0
4,1,3,1,1,1.0,2.0,2.0,1.0,2.0,2.0,991,0,5,91.0


Dealing with missing values and NaN type stuff/weird categories

First, let's check how many
3's in 'IFATHER'
A '3' response means...

In [33]:
#How many 3's in IFATHER? Delete?
#A '3' response indicates that the youth
#does not know if their father is in the HH

(df['IFATHER'] == 3).sum()

np.int64(73)

Since there are only 73 rows that have a response of '3' for 'IFATHER' in the dataset of 10,561 rows, let's remove those 73 rows. This simplifies our model by turning the predictor variable 'IFATHER' into a binary variable instead of categorical. To clarify, a value of '1' indicates the youth's father is in the household and a value of '2' indicates the youth's father is not in the household.

In [37]:
df = df[df['IFATHER'] != 3]

Checking for NaNs/Missing Data

In [42]:
df.isna().sum()

IFATHER       0
POVERTY3      0
IRSEX         0
NEWRACE2      0
PRCHORE2     33
PRLMTTV2     65
PARCHKHW     75
YFLTMRJ2     87
RLGATTD     284
AVGGRADE    708
IRALCAGE      0
MRJFLAG       0
ALCMDAYS      0
IRCIGFM       0
dtype: int64

If it is less than 100 rows (approx. less than 1% of the dataset), delete the Nan rows. 
More than 100?
Use median/mean/mode for others?

Delete rows with nans in these columns
PRCHORE2
PRLMTTV2
PARCHKHW
YFLTMRJ2

Rows with nans in these columns, fill in nans with mode of the column
RLGATTD (Mode)
AVGGRADE (Mode)

In [45]:
#Delete some Nans
rows_nan_delete = ['PRCHORE2', 'PRLMTTV2', 'PARCHKHW', 'YFLTMRJ2']

df = df.dropna(subset=rows_nan_delete)

In [49]:
#Impute Mode
rows_mode_impute = ['RLGATTD', 'AVGGRADE']

for col in rows_mode_impute:
    df[col] = df[col].fillna(df[col].mode()[0])

IFATHER     0
POVERTY3    0
IRSEX       0
NEWRACE2    0
PRCHORE2    0
PRLMTTV2    0
PARCHKHW    0
YFLTMRJ2    0
RLGATTD     0
AVGGRADE    0
IRALCAGE    0
MRJFLAG     0
ALCMDAYS    0
IRCIGFM     0
dtype: int64

Final Dataset to Model with

In [51]:
df.head()

,IFATHER,POVERTY3,IRSEX,NEWRACE2,PRCHORE2,PRLMTTV2,PARCHKHW,YFLTMRJ2,RLGATTD,AVGGRADE,IRALCAGE,MRJFLAG,ALCMDAYS,IRCIGFM
0,1,1,1,7,1.0,2.0,1.0,1.0,2.0,2.0,991,0,5,91.0
1,1,1,2,1,2.0,2.0,1.0,1.0,2.0,2.0,991,1,5,91.0
2,1,3,1,6,1.0,1.0,1.0,1.0,2.0,2.0,11,0,5,91.0
3,1,2,2,7,2.0,2.0,1.0,1.0,2.0,2.0,991,0,5,91.0
4,1,3,1,1,1.0,2.0,2.0,1.0,2.0,2.0,991,0,5,91.0


In [52]:
df.isna().sum()

IFATHER     0
POVERTY3    0
IRSEX       0
NEWRACE2    0
PRCHORE2    0
PRLMTTV2    0
PARCHKHW    0
YFLTMRJ2    0
RLGATTD     0
AVGGRADE    0
IRALCAGE    0
MRJFLAG     0
ALCMDAYS    0
IRCIGFM     0
dtype: int64

In [53]:
df.shape

(10296, 14)

In [56]:
df.dtypes

IFATHER       int64
POVERTY3      int64
IRSEX         int64
NEWRACE2      int64
PRCHORE2    float64
PRLMTTV2    float64
PARCHKHW    float64
YFLTMRJ2    float64
RLGATTD     float64
AVGGRADE    float64
IRALCAGE      int64
MRJFLAG       int64
ALCMDAYS      int64
IRCIGFM     float64
dtype: object

Predictors:
- IFATHER (binary) -> LB
- POVERTY3 (categorical) -> OH
- IRSEX (binary) -> LB
- NEWRACE2 (categorical) -> OH
- PRCHORE2 (binary) -> LB
- PRLMTTV2 (binary) -> LB
- PARCHKHW (binary) -> LB
- YFFLTMRJ2 (binary) -> LB
- RLGATTD (binary) -> LB
- AVGGRADE (binary) -> LB
- IRALCAGE (numeric? actually categorical?) -> OH

Exploring which predictors to one-hot encode (OH) and which to bin (LB).

'IRALCAGE' has 18 different values.

One-hot encode 'IRALCAGE'

In [58]:
df['IRALCAGE'].nunique()

18

In [60]:
df['IRALCAGE'].value_counts()

IRALCAGE
991    7941
15      497
14      471
13      371
16      291
12      255
11      133
17      101
10       75
9        46
7        31
8        24
6        22
5        13
4        12
3         5
2         5
1         3
Name: count, dtype: int64

In [62]:
#LabelBinarizer for some columns
cols_to_bin = ['IFATHER', 'IRSEX', 'PRCHORE2', 'PRLMTTV2', 'PARCHKHW', 'YFLTMRJ2', 'RLGATTD', 'AVGGRADE']

lb = LabelBinarizer()

for col in cols_to_bin:
    df[col] = lb.fit_transform(df[col])

In [64]:
#One-Hot Encoding
cols_to_oh = ['POVERTY3', 'NEWRACE2', 'IRALCAGE']

encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoded_array = encoder.fit_transform(df[cols_to_oh])

#turn back into DataFrame
encoded_df = pd.DataFrame(
    encoded_array,
    columns=encoder.get_feature_names_out(cols_to_oh)
)

#sparse_output=False prevents memory efficient storage
#that prevents easy conversion back into a dataframe

Compare the final cleaned df to youth_data at the start...shape, nans, etc.

Binary Classification Model

Multi-Class Classification Model

Regression Model

Ensemble Methods (like Random Forest, etc.)

Tree Visualization & Interpretation

Discussion Parts Go Here

Conclusion